# YuE2 on Kaggle — restored native FP16 engine (revision 3)

Select **GPU T4 ×2** and enable Internet. This restores the original
`yue2_infer 0.1.5` engine: FP16 transformer on GPU 0, CUDA-graph AR decoding
with SDPA, and tiled FP32 VAE on GPU 1. Default generation uses **Direct to
audio**, **16 Euler steps**, CFG **1.0**, and a **3-minute maximum**.
The duration menu includes **7 minutes** and **8 minutes**.

**No diagnostic song runs automatically.** Run setup and open the UI,
enter your full song lyrics and style, choose your duration, then click
Generate Song. Optional score planning remains off unless you select it.
Cover transcription is disabled in this version.

Durations are maximums. A natural early ending or token cutoff is reported.
Long acoustic generations may span multiple chunks; listen at the reported
boundaries. Seven/eight-minute performance and continuity remain unverified.

Runtime and model files use session-only `/kaggle/temp/yue2_native`.
Song audio, exact tokens and recovery latents use
`/kaggle/working/yue2_native/runs`. The paths may share underlying storage;
setup prints actual free space rather than assuming a separate 50 GB quota.
Old notebook caches and files are preserved.

**After generation, download the complete run ZIP in the UI, or Save
Version with outputs in Kaggle before ending the session.** A saved draft
or a notebook download containing logs does not preserve audio files.


In [ ]:
import base64, json, os, sys, zlib
from pathlib import Path
if "app" in globals():
    app.stop()  # Release a managed server from the earlier ComfyUI notebook.
if "native_process" in globals() and native_process.poll() is None:
    native_process.terminate()
    try:
        native_process.wait(timeout=20)
    except Exception:
        native_process.kill()
        native_process.wait(timeout=10)
support = Path("/kaggle/temp/yue2_native_support")
support.mkdir(parents=True, exist_ok=True)
payload = "..."
for relative, contents in json.loads(zlib.decompress(base64.b64decode(payload))).items():
    path = support / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(contents)
for name in list(sys.modules):
    if name.split(".")[0] in {"native_yue2", "kaggle_yue2"}:
        del sys.modules[name]
if str(support) not in sys.path:
    sys.path.insert(0, str(support))
from native_yue2.setup import install
ROOT = "/kaggle/temp/yue2_native"
OUTPUTS = "/kaggle/working/yue2_native/runs"
native_python, native_env = install(ROOT, OUTPUTS)
print("Setup completed. Run the next cell to open the original native engine UI.")


**Open the generator.** This cell downloads the original full-precision
model assets on first launch and prints a Gradio share link. Open that link.
The cell stays running while the UI is available. Reuse it for additional
songs so the transformer can stay loaded. A stopped cell ends this process.

In the UI select **Full Song - 7 min** or **Full Song - 8 min** when ready.
Supply complete lyrics with sections, instrumental passages and an outro.
The original generation defaults are restored; no speed is promised for
long songs. The status panel reports actual AR execution and stage times.


In [ ]:
import subprocess
engine = support / "native_yue2/engine.py"
log_path = Path(OUTPUTS).parent / "runtime/engine.log"
log_path.parent.mkdir(parents=True, exist_ok=True)
native_process = subprocess.Popen(
    [native_python, "-u", str(engine)], env=native_env,
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
try:
    with log_path.open("a") as log:
        for line in native_process.stdout:
            log.write(line)
            log.flush()
            print(line, end="", flush=True)
    if native_process.wait() != 0:
        raise RuntimeError(f"Native engine stopped. Inspect {log_path}")
finally:
    if native_process.poll() is None:
        native_process.terminate()
        try:
            native_process.wait(timeout=20)
        except subprocess.TimeoutExpired:
            native_process.kill()
            native_process.wait(timeout=10)


**Find and preserve your song.** Each attempt has its own folder. After a
successful generation the UI offers WAV, FLAC, and a complete ZIP containing
settings, tokens, timings and recovery latents. Download it before stopping.
The folder also appears under Kaggle Output at `yue2_native/runs`.
`status.json` records the measured audio length and any cutoff; it does not
establish musical completeness. Listen to the ending and chunk boundaries.

Run the next cell after stopping the UI to list all saved attempts. It
starts no model or generation. Files still need to be downloaded or saved
with a Kaggle output version before the session is discarded.


In [ ]:
from IPython.display import FileLink, display
for directory in sorted(Path(OUTPUTS).glob("*")):
    if directory.is_dir():
        status = directory / "status.json"
        print(directory.name, status.read_text() if status.exists() else "Partial run")
        for filename in ("audio.wav", "audio.flac"):
            if (directory / filename).exists():
                display(FileLink(str(directory / filename)))
for archive in sorted(Path(OUTPUTS).glob("*.zip")):
    display(FileLink(str(archive)))


**Optional decode recovery.** If audio decoding/export failed after
`latent.npy` and its manifest were saved, stop the UI first and point the
next cell at that run folder. This checks the latent checksum and original
model weights, and decodes to a new folder without regenerating music
tokens or the acoustic latents. For a new session, attach the saved run ZIP
as a Kaggle input and extract it first. Setup and the original model files
are still required. ComfyUI artifacts from revision 2 use a different format.


In [ ]:
RECOVER_RUN = None  # Folder containing latent.npy and latent_manifest.json
if RECOVER_RUN:
    if "native_process" in globals() and native_process.poll() is None:
        raise RuntimeError("Stop the UI cell before decoding a saved run")
    recovery_env = dict(native_env, YUE2_RECOVER_RUN=str(Path(RECOVER_RUN).resolve()))
    subprocess.run([native_python, "-u", str(support / "native_yue2/engine.py")],
                   env=recovery_env, check=True)


Based on the original AIQUEST Academy notebook in Movies commit
`fcf4d45dc5dbca57b39e54311807410cdf41f737`.
Sources: [YuE2-3B](https://huggingface.co/m-a-p/YuE2-3B),
[YuE2 VAE](https://huggingface.co/m-a-p/YuE2-Vae).
The native inference wheel and model revisions are pinned; wheel and
model checksums are verified. Upstream model/license terms apply.
Source and checks are included in the Music repository for review.
